In [6]:
import numpy as np
import pandas as pd
import librosa as lib
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix,multilabel_confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import coo_matrix
from pathlib import Path
import shutil

## Importar los rttm y convertirlos en un df

In [7]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm

# ============================================================
# Config
# ============================================================

BASE_DIR = Path("Corpus/transcripciones-segunda-vuelta/highVol")
OUT_DIR = Path("outputs/transcripciones-segunda-vuelta/highVol")
OUT_DIR.mkdir(parents=True, exist_ok=True)

STEP = 0.01  # tu resolución temporal

# ============================================================
# Helpers de lectura RTTM (tus funciones)
# ============================================================
def read_rttm(file_path):
    columns = ['Type', 'File ID', 'Channel', 'Start Time', 'Duration', 'Ortho', 'Ortho1', 'SType', 'Conf']
    df = pd.read_csv(file_path, delim_whitespace=True, header=None, names=columns)
    df['Start Time'] = df['Start Time'].astype(float)
    df['Duration'] = df['Duration'].astype(float)
    df['Conf'] = df['Conf'].astype(float)
    return df

def read_rttm_diar(file_path):
    columns = ['Type', 'File ID', 'Channel', 'Start Time', 'Duration', 'Ortho1', 'Ortho2', 'SType', 'Name1', 'Name2']
    df = pd.read_csv(file_path, delim_whitespace=True, header=None, names=columns)
    df['Start Time'] = df['Start Time'].astype(float)
    df['Duration'] = df['Duration'].astype(float)
    return df

def merge_intervals(iv):
    if not iv:
        return iv
    iv_sorted = sorted(iv, key=lambda x: x[0])
    merged = [iv_sorted[0]]
    for s, e in iv_sorted[1:]:
        last_s, last_e = merged[-1]
        if s <= last_e:  # solapado o contiguo
            merged[-1] = (last_s, max(last_e, e))
        else:
            merged.append((s, e))
    return merged

# ============================================================
# Preparar el mapeo de archivos: base -> {elan, diar}
#   - "elan"  = *.rttm sin sufijo "-diarization"
#   - "diar"  = *-diarization.rttm
# ============================================================
pairs = {}
for f in BASE_DIR.glob("*.rttm"):
    fn = f.name
    if fn.endswith("-diarization.rttm"):
        base = fn[:-len("-diarization.rttm")]  # quita el sufijo
        pairs.setdefault(base, {})["diar"] = f
    else:
        base = fn[:-len(".rttm")]             # base sin extensión
        pairs.setdefault(base, {})["elan"] = f

# ============================================================
# Función que procesa un par (elan + diar) y devuelve df_results
# ============================================================
def process_pair(base_name, elan_path: Path, diar_path: Path, step=STEP):
    # ---------------- Leer
    df_rttm = read_rttm(elan_path)
    df_rttm_diar = read_rttm_diar(diar_path)

    # ---------------- Mapear SType -> STypeNew (Elan)
    stype = df_rttm["SType"].astype(str).str.strip().str.upper()
    
    conditions = [
        stype.eq("CHI"),
        stype.eq("CODE"),
        stype.str.startswith(("FC", "MC")),
        stype.str.startswith("MA"),
        stype.str.startswith("F")
    ]
    choices = ["KCHI", "code", "OCH", "MAL", "FEM"]
    
    df_rttm["STypeNew"] = np.select(conditions, choices, default=pd.NA)
    # ---------------- STypeNew para diar
    df_rttm_diar["STypeNew"] = np.where(
    df_rttm_diar["SType"] == "CHI",
    "OCH",
    df_rttm_diar["SType"]
    )

    # ---------------- Tipificar 'Type' y columnas mínimas
    df_rttm['Type'] = 'Elan'
    df_rttm_diar['Type'] = 'Diar'

    df_rttm = df_rttm[['Type', 'File ID', 'Start Time', 'Duration', 'STypeNew']]
    df_rttm_diar = df_rttm_diar[['Type', 'File ID', 'Start Time', 'Duration', 'STypeNew']]

    # ---------------- Concatenar ambos
    df_both = pd.concat([df_rttm, df_rttm_diar], ignore_index=True)
    df_both = df_both.dropna(subset=['STypeNew'])

    # ---------------- End Time (acelera filtros)
    if "End Time" not in df_both.columns:
        df_both["End Time"] = df_both["Start Time"] + df_both["Duration"]

    # ---------------- Filtrar "code" en Elan para construir intervalos
    #     (si no hay "code", se procesará 0 pasos)
    df_rttm["Start Time"] = pd.to_numeric(df_rttm["Start Time"], errors="coerce")
    df_rttm["Duration"] = pd.to_numeric(df_rttm["Duration"], errors="coerce")

    df_code = (
        df_rttm.loc[df_rttm["STypeNew"].eq("code"), ["Start Time", "Duration"]]
               .dropna()
               .query("`Duration` > 0")
               .sort_values("Start Time", kind="stable")
    )

    intervals = list(
        df_code.apply(lambda r: (float(r["Start Time"]), float(r["Start Time"] + r["Duration"])), axis=1)
    )
    intervals = merge_intervals(intervals)

    # ---------------- Si no hay intervalos "code", devolvemos vacío con metadata
    if not intervals:
        return pd.DataFrame({
            'Elan_KCHI': [], 'Elan_OCH': [], 'Elan_FEM': [], 'Elan_MAL': [], 'Elan_ELE': [],
            'Diar_KCHI': [], 'Diar_OCH': [], 'Diar_FEM': [], 'Diar_MAL': [], 'Diar_SPEECH': []
        })

    # ---------------- Inicializar resultados
    results = {
        'Elan_KCHI': [], 'Elan_OCH': [], 'Elan_FEM': [], 'Elan_MAL': [], 'Elan_ELE': [],
        'Diar_KCHI': [], 'Diar_OCH': [], 'Diar_FEM': [], 'Diar_MAL': [], 'Diar_SPEECH': []
    }

    # ---------------- Calcular total de pasos
    total_steps = 0
    for s, e in intervals:
        if e > s:
            total_steps += int(np.ceil((e - s) / step))

    # ---------------- Iterar con barra de progreso
    with tqdm(total=total_steps, desc=f"{base_name}: Processing intervals ({len(intervals)} blocks)") as pbar:
        for s, e in intervals:
            if e <= s:
                continue
            # Generamos una vista para reducir scans:
            # (Esto aún es simple; si querés más performance se puede indexar por tiempo)
            for i in np.arange(s, e, step):
                for key in results.keys():
                    type_name, stype_new = key.split('_', 1)  # 'Elan' / 'Diar', 'KCHI'/...
                    has_any = (
                        df_both[
                            (df_both['Start Time'] <= i) &
                            (df_both['End Time'] > i) &
                            (df_both['STypeNew'] == stype_new) &
                            (df_both['Type'] == type_name)
                        ].shape[0] > 0
                    )
                    results[key].append(int(has_any))
                pbar.update(1)

    df_results = pd.DataFrame(results)
    return df_results

# ============================================================
# Loop general sobre todos los pares encontrados
# ============================================================
all_outputs = []  # para concatenar si querés un único CSV global

print(f"Se encontraron {len(pairs)} posibles bases en {BASE_DIR.resolve()}\n")

for base_name, d in pairs.items():
    if "elan" not in d or "diar" not in d:
        # Aviso si falta alguno de los dos archivos
        faltante = "elan" if "elan" not in d else "diar"
        print(f"[AVISO] Salteando '{base_name}': falta archivo {faltante}.")
        continue

    elan_path = d["elan"]
    diar_path = d["diar"]

    print(f"Procesando: {base_name}")
    df_results = process_pair(base_name, elan_path, diar_path, step=STEP)

    # Guardar cada resultado individual
    out_file = OUT_DIR / f"{base_name}_results.csv"
    df_results.to_csv(out_file, index=False)
    print(f" -> Guardado: {out_file}")

    # (Opcional) agregar metadata para concatenado global
    if not df_results.empty:
        df_results["name"] = base_name
        all_outputs.append(df_results)

# ============================================================
# (Opcional) Un único CSV global con todos los archivos
# ============================================================
if all_outputs:
    df_all = pd.concat(all_outputs, ignore_index=True)
#     df_all.to_csv(OUT_DIR / "_all_results.csv", index=False)
    print(f"\nArchivo global: {OUT_DIR / '_all_results.csv'}")
else:
    print("\nNo hubo resultados (posiblemente no hay intervalos 'code' en Elan).")


C:\Users\pablo\AppData\Local\Temp\ipykernel_34228\2993474309.py:22: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file_path, delim_whitespace=True, header=None, names=columns)
C:\Users\pablo\AppData\Local\Temp\ipykernel_34228\2993474309.py:30: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file_path, delim_whitespace=True, header=None, names=columns)


Se encontraron 3 posibles bases en C:\Users\pablo\Documents\CIIPME\Pablo\tesis_pablo\Corpus\transcripciones-segunda-vuelta\highVol

Procesando: brandona-a1-nsb


brandona-a1-nsb: Processing intervals (39 blocks): 100%|██████████| 600000/600000 [2:47:11<00:00, 59.81it/s]  
C:\Users\pablo\AppData\Local\Temp\ipykernel_34228\2993474309.py:22: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file_path, delim_whitespace=True, header=None, names=columns)
C:\Users\pablo\AppData\Local\Temp\ipykernel_34228\2993474309.py:30: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file_path, delim_whitespace=True, header=None, names=columns)


 -> Guardado: outputs\transcripciones-segunda-vuelta\highVol\brandona-a1-nsb_results.csv
Procesando: jeremiasc-a1-nsm


jeremiasc-a1-nsm: Processing intervals (28 blocks): 100%|██████████| 360000/360000 [1:00:43<00:00, 98.80it/s] 


 -> Guardado: outputs\transcripciones-segunda-vuelta\highVol\jeremiasc-a1-nsm_results.csv
[AVISO] Salteando 'valentina-a1-nsb': falta archivo diar.

Archivo global: outputs\transcripciones-segunda-vuelta\highVol\_all_results.csv


## Crear directorios por chico para guardar todo

In [8]:
# Carpeta donde quedaron los CSV ya generados
source_dir = Path("outputs/transcripciones-segunda-vuelta/highVol")

# Carpeta base de salida reorganizada por niño
target_base_dir = Path("outputs/por_chico")
target_base_dir.mkdir(parents=True, exist_ok=True)

# Buscar solo archivos *_results.csv, excluyendo el global
csv_paths = sorted(
    p for p in source_dir.glob("*_results.csv")
    if p.name != "_all_results.csv"
)

print("Archivos encontrados:")
for p in csv_paths:
    print(p)

print(f"\nTotal: {len(csv_paths)} archivos")

Archivos encontrados:
outputs\transcripciones-segunda-vuelta\highVol\brandona-a1-nsb_results.csv
outputs\transcripciones-segunda-vuelta\highVol\jeremiasc-a1-nsm_results.csv

Total: 2 archivos


In [9]:
source_dir = Path("outputs/transcripciones-segunda-vuelta/highVol")
target_base_dir = Path("outputs/transcripciones-segunda-vuelta/highVol")
target_base_dir.mkdir(parents=True, exist_ok=True)

csv_paths = sorted(
    p for p in source_dir.glob("*_results.csv")
    if p.name != "_all_results.csv"
)

for csv_path in csv_paths:
    child_name = csv_path.stem.replace("_results", "")
    child_dir = target_base_dir / child_name
    child_dir.mkdir(parents=True, exist_ok=True)

    df_results = pd.read_csv(csv_path)
    df_results.to_csv(child_dir / f"df_results_{child_name}.csv", index=False)

    print(f"OK -> {child_dir}")

OK -> outputs\transcripciones-segunda-vuelta\highVol\brandona-a1-nsb
OK -> outputs\transcripciones-segunda-vuelta\highVol\jeremiasc-a1-nsm


## Crear metricas, exportarlas y graficos por cada chico

In [10]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
    multilabel_confusion_matrix,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

# =========================================================
# CONFIGURACIÓN GENERAL
# =========================================================
source_dir = Path("outputs/transcripciones-segunda-vuelta/highVol")
target_base_dir = Path("outputs/transcripciones-segunda-vuelta/highVol")
target_base_dir.mkdir(parents=True, exist_ok=True)

labels = ['KCHI', 'OCH', 'FEM', 'MAL', 'SIL']
types = ['KCHI', 'OCH', 'FEM', 'MAL', 'SIL']

sns.set(style="whitegrid")

csv_paths = sorted(
    p for p in source_dir.glob("*_results.csv")
    if p.name != "_all_results.csv"
)

print(f"Se encontraron {len(csv_paths)} archivos para procesar.\n")


# =========================================================
# FUNCIÓN AUXILIAR
# Convierte multilabel a singlelabel tomando la primera clase activa
# Si ninguna clase está activa, devuelve -1
# =========================================================
def multilabel_to_singlelabel(y_multilabel):
    single_labels = []
    for row in y_multilabel:
        indices = np.where(row == 1)[0]
        if len(indices) == 0:
            single_labels.append(-1)
        else:
            single_labels.append(indices[0])
    return np.array(single_labels)


# =========================================================
# LOOP PRINCIPAL
# =========================================================
for csv_path in csv_paths:
    name = csv_path.stem.replace("_results", "")
    print(f"Procesando: {name}")

    # -----------------------------------------------------
    # Crear carpetas por chico
    # -----------------------------------------------------
    child_dir = target_base_dir / name
    csv_dir = child_dir / "csv"
    graficos_dir = child_dir / "graficos"
    reportes_dir = child_dir / "reportes"

    csv_dir.mkdir(parents=True, exist_ok=True)
    graficos_dir.mkdir(parents=True, exist_ok=True)
    reportes_dir.mkdir(parents=True, exist_ok=True)

    # -----------------------------------------------------
    # Leer CSV original
    # -----------------------------------------------------
    df_resultados = pd.read_csv(csv_path)

    # Guardar copia del CSV base
    df_resultados.to_csv(csv_dir / f"df_results_{name}.csv", index=False)

    # =====================================================
    # CREAR SIL SIN INCLUIR ELE EN EL ANÁLISIS
    # SIL en Elan: 1 cuando KCHI/OCH/FEM/MAL están todos en 0
    # SIL en Diar: negación de SPEECH
    # =====================================================
    cols_elan_main = ["Elan_KCHI", "Elan_OCH", "Elan_FEM", "Elan_MAL"]

    df_resultados["Elan_SIL"] = (
        df_resultados[cols_elan_main].eq(0).all(axis=1)
    ).astype(int)

    df_resultados["Diar_SIL"] = 1 - df_resultados["Diar_SPEECH"]

    # Guardar también la versión enriquecida con SIL
    df_resultados.to_csv(csv_dir / f"df_results_{name}.csv", index=False)

    # =====================================================
    # PARTE 1 - MATRICES MULTILABEL + CLASSIFICATION REPORT
    # =====================================================
    y_true_multi = df_resultados[
        ['Elan_KCHI', 'Elan_OCH', 'Elan_FEM', 'Elan_MAL', 'Elan_SIL']
    ].values

    y_pred_multi = df_resultados[
        ['Diar_KCHI', 'Diar_OCH', 'Diar_FEM', 'Diar_MAL', 'Diar_SIL']
    ].values

    conf_matrix = multilabel_confusion_matrix(y_true_multi, y_pred_multi)

    report_lines = []
    for i, label in enumerate(labels):
        report_lines.append(f"Matriz de confusión para la etiqueta {label}:")
        report_lines.append(str(conf_matrix[i]))
        report_lines.append("")

    class_report = classification_report(
        y_true_multi,
        y_pred_multi,
        target_names=labels,
        zero_division=0
    )

    report_lines.append("Informe de clasificación:")
    report_lines.append(class_report)

    with open(reportes_dir / f"classification_report_{name}.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(report_lines))

    # =====================================================
    # PARTE 2 - MATRIZ DE CONFUSIÓN NORMALIZADA
    # =====================================================
    y_true_single = multilabel_to_singlelabel(y_true_multi)
    y_pred_single = multilabel_to_singlelabel(y_pred_multi)

    valid_indices = y_true_single != -1
    y_true_single_valid = y_true_single[valid_indices]
    y_pred_single_valid = y_pred_single[valid_indices]

    if len(y_true_single_valid) > 0:
        cm = confusion_matrix(
            y_true_single_valid,
            y_pred_single_valid,
            labels=range(len(labels)),
            normalize='true'
        )

        cm_df = pd.DataFrame(cm, index=labels, columns=labels)
        annot = cm_df.map(lambda x: f"{x*100:.1f}%")

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm_df, annot=annot, fmt='', cmap='Blues', cbar=False)
        plt.title(f'Matriz de Confusión Normalizada\n{name}')
        plt.ylabel('Etiquetas Verdaderas (Elan)')
        plt.xlabel('Etiquetas Predichas (Diar)')
        plt.tight_layout()
        plt.savefig(graficos_dir / f"matriz_confusion_{name}.png", dpi=300, bbox_inches='tight')
        plt.close()

    # =====================================================
    # PARTE 3 - MÉTRICAS POR TIPO + AVERAGE
    # =====================================================
    metrics = {
        'Tipo': [],
        'Precisión': [],
        'Recall': [],
        'F1-score': []
    }

    for t in types:
        y_true = df_resultados['Elan_' + t]
        y_pred = df_resultados['Diar_' + t]

        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        metrics['Tipo'].append(t)
        metrics['Precisión'].append(precision)
        metrics['Recall'].append(recall)
        metrics['F1-score'].append(f1)

    df_metrics = pd.DataFrame(metrics)

    avg_row = pd.DataFrame([{
        'Tipo': 'Average',
        'Precisión': df_metrics['Precisión'].mean(),
        'Recall': df_metrics['Recall'].mean(),
        'F1-score': df_metrics['F1-score'].mean()
    }])

    df_metrics_export = pd.concat([df_metrics, avg_row], ignore_index=True)
    df_metrics_export.to_csv(csv_dir / f"df_metrics_{name}.csv", index=False)

    # =====================================================
    # PARTE 4 - GRÁFICOS DE MÉTRICAS
    # =====================================================
    df_metrics_plot = df_metrics.copy()

    # Precisión
    plt.figure(figsize=(8, 6))
    sns.barplot(x='Tipo', y='Precisión', data=df_metrics_plot, palette='Blues_d')
    plt.title(f'Precisión por Tipo\n{name}')
    plt.ylim(0, 1)
    plt.ylabel('Precisión')
    plt.tight_layout()
    plt.savefig(graficos_dir / f"precision_{name}.png", dpi=300, bbox_inches='tight')
    plt.close()

    # Recall
    plt.figure(figsize=(8, 6))
    sns.barplot(x='Tipo', y='Recall', data=df_metrics_plot, palette='Greens_d')
    plt.title(f'Recall por Tipo\n{name}')
    plt.ylim(0, 1)
    plt.ylabel('Recall')
    plt.tight_layout()
    plt.savefig(graficos_dir / f"recall_{name}.png", dpi=300, bbox_inches='tight')
    plt.close()

    # F1-score
    plt.figure(figsize=(8, 6))
    sns.barplot(x='Tipo', y='F1-score', data=df_metrics_plot, palette='Reds_d')
    plt.title(f'F1-score por Tipo\n{name}')
    plt.ylim(0, 1)
    plt.ylabel('F1-score')
    plt.tight_layout()
    plt.savefig(graficos_dir / f"f1score_{name}.png", dpi=300, bbox_inches='tight')
    plt.close()

    # Combinado
    df_metrics_melted = pd.melt(
        df_metrics_plot,
        id_vars=['Tipo'],
        value_vars=['Precisión', 'Recall', 'F1-score'],
        var_name='Métrica',
        value_name='Valor'
    )

    plt.figure(figsize=(10, 6))
    sns.barplot(x='Tipo', y='Valor', hue='Métrica', data=df_metrics_melted)
    plt.title(f'Métricas por Tipo\n{name}')
    plt.ylim(0, 1)
    plt.ylabel('Valor')
    plt.legend(title='Métrica')
    plt.tight_layout()
    plt.savefig(graficos_dir / f"combinado_{name}.png", dpi=300, bbox_inches='tight')
    plt.close()

    # =====================================================
    # PARTE 5 - GRÁFICOS ESPECIALES DE ESTADOS
    # PARA KCHI / OCH / FEM / MAL / SIL
    # =====================================================
    df_plot = df_resultados.copy()

    # Separar visualmente Diar de Elan
    diar_cols_plot = [col for col in df_plot.columns if col.startswith('Diar_')]
    df_plot[diar_cols_plot] = df_plot[diar_cols_plot].replace(1, 0.8)

    types_plot = ['KCHI', 'OCH', 'FEM', 'MAL', 'SIL']
    colors = ['#FF6347', '#4682B4']  # Elan / Diar

    # Si tenés una columna de tiempo real, usala acá:
    # time_col = "time"
    # x_values = df_plot[time_col]

    # Si no, usamos el índice
    x_values = df_plot.index

    for t in types_plot:
        elan_col = f'Elan_{t}'
        diar_col = f'Diar_{t}'

        plt.figure(figsize=(16, 6))

        subset_elan = df_plot[df_plot[elan_col] > 0]
        subset_diar = df_plot[df_plot[diar_col] > 0]

        plt.scatter(subset_elan.index, subset_elan[elan_col], label=elan_col, color=colors[0], s=8)
        plt.scatter(subset_diar.index, subset_diar[diar_col], label=diar_col, color=colors[1], s=8)

        plt.title(f'Estados de {elan_col} y {diar_col}\n{name}')
        plt.xlabel('Tiempo [seg]')
        plt.ylabel('Valores Binarios')
        plt.legend(loc='upper right')
        plt.grid(True)

        # Toda la duración del archivo
        plt.xlim(df_plot.index.min(), df_plot.index.max())

        # Si querés usar una ventana específica, descomentá esto:
        # plt.xlim(1000, 1400)

        plt.tight_layout()
        plt.savefig(graficos_dir / f"estados_{t.lower()}_{name}.png", dpi=300, bbox_inches='tight')
        plt.close()

    # =====================================================
    # CHEQUEOS ÚTILES
    # =====================================================
    resumen_chequeos = []
    resumen_chequeos.append(f"Archivo: {name}")
    resumen_chequeos.append("")
    resumen_chequeos.append("Conteo Elan_SIL:")
    resumen_chequeos.append(str(df_resultados["Elan_SIL"].value_counts(dropna=False)))
    resumen_chequeos.append("")
    resumen_chequeos.append("Conteo Diar_SIL:")
    resumen_chequeos.append(str(df_resultados["Diar_SIL"].value_counts(dropna=False)))
    resumen_chequeos.append("")

    with open(reportes_dir / f"chequeos_{name}.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(resumen_chequeos))

    print(f"OK -> {child_dir}\n")

print("Proceso finalizado.")

Se encontraron 2 archivos para procesar.

Procesando: brandona-a1-nsb


C:\Users\pablo\AppData\Local\Temp\ipykernel_34228\1146975637.py:199: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='Tipo', y='Precisión', data=df_metrics_plot, palette='Blues_d')
C:\Users\pablo\AppData\Local\Temp\ipykernel_34228\1146975637.py:209: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='Tipo', y='Recall', data=df_metrics_plot, palette='Greens_d')
C:\Users\pablo\AppData\Local\Temp\ipykernel_34228\1146975637.py:219: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='Tipo', y='F1-score', data=df_metrics_plot, palette='Reds_d')


OK -> outputs\transcripciones-segunda-vuelta\highVol\brandona-a1-nsb

Procesando: jeremiasc-a1-nsm


C:\Users\pablo\AppData\Local\Temp\ipykernel_34228\1146975637.py:199: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='Tipo', y='Precisión', data=df_metrics_plot, palette='Blues_d')
C:\Users\pablo\AppData\Local\Temp\ipykernel_34228\1146975637.py:209: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='Tipo', y='Recall', data=df_metrics_plot, palette='Greens_d')
C:\Users\pablo\AppData\Local\Temp\ipykernel_34228\1146975637.py:219: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='Tipo', y='F1-score', data=df_metrics_plot, palette='Reds_d')


OK -> outputs\transcripciones-segunda-vuelta\highVol\jeremiasc-a1-nsm

Proceso finalizado.
